<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/social_media/Final_tweet_Analytics_DS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Kwanda Mazibuko** - stdnr: 1077167

# **Importing Libraries**

In [2]:
!pip install -q tweepy gensim python-louvain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.2 MB/s eta 0:00:00


In [3]:
# Importing Libraries - make sure the packages are installed
import os
import tweepy as tw
import re
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import BytesIO

import networkx as nx
import community as community_louvain
from community.community_louvain import best_partition

import warnings
warnings.filterwarnings("ignore")

#making sure that we can see all rows and cols
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# **Loading Data**

In [4]:
# Reading data
%%time
FILE_ID = "1lhoPhOLB4fm-IgXlKhZFfC0UiFfHSklZ"
xlsx_url = f"https://docs.google.com/spreadsheets/d/{FILE_ID}/export?format=xlsx"
r = requests.get(xlsx_url)
df_1 = pd.read_excel(BytesIO(r.content), engine = "openpyxl")


CPU times: user 1min 46s, sys: 675 ms, total: 1min 47s
Wall time: 1min 52s


In [5]:
df_1.head(1)

,Query Id,Query Name,Date,Title,Url,Domain,Sentiment,Page Type,Language,Country Code,Continent Code,Continent,Country,City Code,Account Type,Added,Assignment,Author,Category Details,Checked,City,Display URLs,Entity Info,Expanded URLs,Facebook Author ID,Facebook Comments,Facebook Likes,Facebook Role,Facebook Shares,Facebook Subtype,Full Name,Full Text,Gender,Impressions,Instagram Comments,Instagram Followers,Instagram Following,Instagram Interactions Count,Instagram Posts,Interest,Last Assignment Date,Latitude,Location Name,Longitude,Media Filter,Media URLs,Mentioned Authors,Original Url,Priority,Professions,Resource Id,Short URLs,Starred,Station Name,Viewership,Status,Subtype,Thread Author,Thread Created Date,Thread Entry Type,Thread Id,Thread URL,Total Monthly Visitors,X Author ID,X Channel Role,X Followers,X Following,X Replies,X Reply to,X Repost of,X Reposts,X Likes,X Posts,X Verified,Updated,Reach (new),Publication Name,Licenses,Redacted,Redacted Fields,Redaction Reason,Asset Content Id,Asset Thumb Id,Author Verified Type,Avatar,Batch Id,Blog Name,Broadcast Media Url,Is Syndicated,Air Type,Broadcast Type,Media Type,Ad Value,Circulation,Region,Region Code,Daily Visitors,Engagement Type,Hashtags,Item Review,Kicker,Linkedin Comments,Linkedin Engagement,Linkedin Impressions,Linkedin Likes,Linkedin Shares,Linkedin Sponsored,Linkedin Video Views,Parent Post Id,Parent Blog Name,Pub Type,Publisher Sub Type,Rating,Reddit Score,Reddit Score Upvote Ratio,Reddit Comments,Reddit Author Karma,Root Post Id,Root Blog Name,Subreddit,Subreddit Subscribers,Subscriptions,Sub Title,React Score Overall,React Score Emotionality,React Score Harmful,Engagement Score,Subreddit NSFW,Reddit Post Flair,Reddit Author Flair,Subreddit Topics,Reddit Spoiler,Publication Id,Page Type Name,Content Source,Content Source Name,Custom,Bluesky Author Id,Bluesky Followers,Bluesky Following,Bluesky Likes,Bluesky Posts,Bluesky Quotes,Bluesky Replies,Bluesky Reposts,Can Edit Markup,Can Edit Metadata,Can Edit Segmentation,Can Edit Workflow,Copyright,Factiva Attribute Code,Has Full Text,Impact,Instagram Likes,Mention Id,Podcast Audience Estimate,Podcast Duration Ms,Raw Metadata,Reportable,Threads Likes,Threads Quotes,Threads Replies,Threads Reposts,Threads Shares,Threads Views,Tiktok Comments,Tiktok Connected Account,Tiktok Likes,Tiktok Reach,Tiktok Shares,Tiktok Views,Weblog Title,Youtube Comments,Youtube Duration Milliseconds,Youtube Favourites,Youtube Likes,Youtube Subscriber Count,Youtube Video Count,Emotion
0,2003594270,Kenya protests 2025,2025-08-31 21:59:50.0,RT @_James041 Aden Duale is as guilty as F.\n\nGuy has tried ethical card and it has failed.\n\nSHA theft has affected everyone and no one wants to be associated with a thief.\n\nHe has tried hiring affordable bloggers and they have all been humbled by the truth.\n\nThe more a crocodile smiles the more his anus widens.\n\nThere's no escape route for the wicked!\n\n#DualeMustGo #RutoMustGo #DrainTheSwamp,http://twitter.com/kelvinngari62/statuses/1962274155270635732,twitter.com,negative,twitter,en,KEN,AFRICA,Africa,Kenya,KEN.Coast.Mombasa,individual,2025-09-02T09:16:47.214+0000,NaN,kelvinngari62,NaN,False,Mombasa,NaN,"{entityId=13414952, entityConfidence=HIGH, url=https://www.wikidata.org/wiki/Q13414952}, {entityId=43169, entityConfidence=MEDIUM, url=https://www.wikidata.org/wiki/Q43169}, {entityId=2727213, entityConfidence=MEDIUM, url=https://www.wikidata.org/wiki/Q2727213}, {entityId=4682154, entityConfidence=LOW, url=https://www.wikidata.org/wiki/Q4682154}, {entityId=497, entityConfidence=LOW, url=https://www.wikidata.org/wiki/Q497}",NaN,NaN,0,0,NaN,0,NaN,kelvinngari62 (knn),RT @_James041 Aden Duale is as guilty as F.\n\nGuy has tried ethical card and it has failed.\n\nSHA theft has affected everyone and no one wants to be associated with a thief.\n\nHe has tried hiring affordable bloggers and they have all been humbled by the truth.\n\nThe more a crocodile smiles the more his anus widens.\n\nThere's no e

In [6]:
df_1.shape

(49823, 179)

#### **Pre-Processing**

In [7]:
# Changing column names
df_1.columns = df_1.columns.str.replace(' ', '_', regex=False).str.lower()

In [8]:
# Convert 'Date' column to datetime format
df_1['tweet_date'] = pd.to_datetime(df_1['date'], errors = 'coerce')

# June 2025 Tweets
df_temp = df_1[(df_1['tweet_date'].dt.year == 2025) & (df_1['tweet_date'].dt.month == 6)]

df_temp.reset_index(drop=True, inplace=True)

# final Data
df_final = df_temp[['author','date','title','full_text','tweet_date', 'sentiment','emotion','account_type','engagement_type','gender']]

In [9]:
df_final.columns

Index(['author', 'date', 'title', 'full_text', 'tweet_date', 'sentiment',
       'emotion', 'account_type', 'engagement_type', 'gender'],
      dtype='object')

# **Question One**





In [16]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

# Load your dataset
df = df_final

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Unique authors: {df['author'].nunique()}")

Dataset loaded successfully!
Shape: (32797, 10)
Columns: ['author', 'date', 'title', 'full_text', 'tweet_date', 'sentiment', 'emotion', 'account_type', 'engagement_type', 'gender']
Unique authors: 12511


In [17]:
def extract_mentions_and_retweets(df):
    """Extract mention and retweet relationships from the data"""

    mention_edges = []
    retweet_edges = []

    for _, row in df.iterrows():
        author = row['author']
        text = str(row['full_text'])

        # Extract mentions
        mentions = re.findall(r'@([a-zA-Z0-9_]+)', text)
        for mentioned_user in mentions:
            if mentioned_user.lower() != author.lower():
                mention_edges.append((author, mentioned_user))

        # Extract retweets
        text_lower = text.lower()
        if text_lower.startswith('rt @') or ' rt @' in text_lower:
            rt_matches = re.findall(r'rt @([a-zA-Z0-9_]+)', text_lower)
            if rt_matches:
                retweeted_user = rt_matches[0]
                if retweeted_user.lower() != author.lower():
                    retweet_edges.append((author, retweeted_user))

    return mention_edges, retweet_edges

# Execute this step
print("Extracting network relationships...")
mention_edges, retweet_edges = extract_mentions_and_retweets(df)

print(f"Found {len(mention_edges)} mention edges")
print(f"Found {len(retweet_edges)} retweet edges")

Extracting network relationships...
Found 34566 mention edges
Found 29761 retweet edges


In [18]:
# Create mention network
G_mention = nx.DiGraph()
G_mention.add_edges_from(mention_edges)

# Create retweet network
G_retweet = nx.DiGraph()
G_retweet.add_edges_from(retweet_edges)

print("Networks created successfully!")
print(f"Mention network: {G_mention.number_of_nodes()} nodes, {G_mention.number_of_edges()} edges")
print(f"Retweet network: {G_retweet.number_of_nodes()} nodes, {G_retweet.number_of_edges()} edges")

Networks created successfully!
Mention network: 13865 nodes, 33389 edges
Retweet network: 13159 nodes, 28763 edges


In [ ]:
def compute_basic_metrics(G, network_name):
    """Compute basic network metrics for a given graph"""

    print(f"\n--- {network_name.upper()} METRICS ---")

    # Basic stats
    print(f"Nodes: {G.number_of_nodes()}")
    print(f"Edges: {G.number_of_edges()}")
    print(f"Density: {nx.density(G):.6f}")

    # Degree centrality
    in_degree = dict(G.in_degree())
    out_degree = dict(G.out_degree())

    # Calculate averages
    if in_degree:
        avg_in_degree = sum(in_degree.values()) / len(in_degree)
        print(f"Average in-degree: {avg_in_degree:.2f}")

    if out_degree:
        avg_out_degree = sum(out_degree.values()) / len(out_degree)
        print(f"Average out-degree: {avg_out_degree:.2f}")

    return {
        'in_degree': in_degree,
        'out_degree': out_degree,
        'betweenness': nx.betweenness_centrality(G),
        'clustering': nx.clustering(G),
        'pagerank': nx.pagerank(G)
    }

# Compute metrics for both networks
print("Computing network metrics...")
mention_metrics = compute_basic_metrics(G_mention, "Mention Network")
retweet_metrics = compute_basic_metrics(G_retweet, "Retweet Network")

Computing network metrics...

--- MENTION NETWORK METRICS ---
Nodes: 13865
Edges: 33389
Density: 0.000174
Average in-degree: 2.41
Average out-degree: 2.41


In [ ]:
def get_top_nodes(metric_dict, metric_name, top_n=15):
    """Get top N nodes from a metric dictionary"""
    return sorted(metric_dict.items(), key=lambda x: x[1], reverse=True)[:top_n]

print("\n=== TOP 15 INFLUENTIAL ACCOUNTS ===")

# Mention network influencers
print("\nMENTION NETWORK - Top 15 by In-Degree:")
top_mention_in_degree = get_top_nodes(mention_metrics['in_degree'], 'in_degree')
for i, (user, score) in enumerate(top_mention_in_degree, 1):
    print(f"{i:2d}. {user}: {score}")

# Retweet network influencers
print("\nRETWEET NETWORK - Top 15 by In-Degree:")
top_retweet_in_degree = get_top_nodes(retweet_metrics['in_degree'], 'in_degree')
for i, (user, score) in enumerate(top_retweet_in_degree, 1):
    print(f"{i:2d}. {user}: {score}")

# Top by betweenness (brokers)
print("\nTOP BROKERS - Mention Network (Betweenness):")
top_mention_betweenness = get_top_nodes(mention_metrics['betweenness'], 'betweenness')
for i, (user, score) in enumerate(top_mention_betweenness, 1):
    print(f"{i:2d}. {user}: {score:.4f}")

Open these .gexf files in Gephi to visualize:
- Use ForceAtlas2 layout
- Color nodes by centrality
- Size nodes by degree
- Label top influencer


In [ ]:
def add_node_attributes(G, df, metrics_dict):
    """Add comprehensive attributes to nodes for Gephi visualization"""

    for node in G.nodes():
        # Network metrics
        G.nodes[node]['degree'] = metrics_dict['in_degree'].get(node, 0)
        G.nodes[node]['betweenness'] = metrics_dict['betweenness'].get(node, 0)
        G.nodes[node]['clustering'] = metrics_dict['clustering'].get(node, 0)
        G.nodes[node]['pagerank'] = metrics_dict['pagerank'].get(node, 0)

        # User attributes from original data
        user_data = df[df['author'] == node]
        if len(user_data) > 0:
            G.nodes[node]['tweet_count'] = len(user_data)
            G.nodes[node]['account_type'] = user_data['account_type'].iloc[0] if 'account_type' in user_data else 'unknown'
            G.nodes[node]['gender'] = user_data['gender'].iloc[0] if 'gender' in user_data else 'unknown'
            if 'sentiment' in user_data:
                G.nodes[node]['avg_sentiment'] = user_data['sentiment'].mean()

# Add attributes and export
print("Adding node attributes...")
add_node_attributes(G_mention, df, mention_metrics)
add_node_attributes(G_retweet, df, retweet_metrics)

print("Exporting to Gephi format...")
nx.write_gexf(G_mention, 'kenyan_protests_mention_network.gexf')
nx.write_gexf(G_retweet, 'kenyan_protests_retweet_network.gexf')

print("✓ Exported mention_network.gexf")
print("✓ Exported retweet_network.gexf")

In [ ]:
def create_simple_plots(mention_metrics, retweet_metrics):
    """Create basic visualizations of network metrics"""

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # Plot 1: Mention network degree distribution
    mention_degrees = list(mention_metrics['in_degree'].values())
    axes[0, 0].hist(mention_degrees, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].set_title('Mention Network - In-Degree Distribution')
    axes[0, 0].set_xlabel('In-Degree')
    axes[0, 0].set_ylabel('Frequency')

    # Plot 2: Retweet network degree distribution
    retweet_degrees = list(retweet_metrics['in_degree'].values())
    axes[0, 1].hist(retweet_degrees, bins=30, alpha=0.7, color='lightcoral', edgecolor='black')
    axes[0, 1].set_title('Retweet Network - In-Degree Distribution')
    axes[0, 1].set_xlabel('In-Degree')
    axes[0, 1].set_ylabel('Frequency')

    # Plot 3: Top mention influencers
    top_mention = get_top_nodes(mention_metrics['in_degree'], 'in_degree', 10)
    users = [user for user, _ in top_mention]
    scores = [score for _, score in top_mention]

    axes[1, 0].barh(range(len(users)), scores, color='skyblue')
    axes[1, 0].set_yticks(range(len(users)))
    axes[1, 0].set_yticklabels(users, fontsize=9)
    axes[1, 0].set_title('Top 10 Mention Influencers')
    axes[1, 0].set_xlabel('In-Degree')

    # Plot 4: Top retweet influencers
    top_retweet = get_top_nodes(retweet_metrics['in_degree'], 'in_degree', 10)
    users = [user for user, _ in top_retweet]
    scores = [score for _, score in top_retweet]

    axes[1, 1].barh(range(len(users)), scores, color='lightcoral')
    axes[1, 1].set_yticks(range(len(users)))
    axes[1, 1].set_yticklabels(users, fontsize=9)
    axes[1, 1].set_title('Top 10 Retweet Influencers')
    axes[1, 1].set_xlabel('In-Degree')

    plt.tight_layout()
    plt.savefig('network_analysis_basic.png', dpi=300, bbox_inches='tight')
    plt.show()

print("Creating visualizations...")
create_simple_plots(mention_metrics, retweet_metrics)

In [ ]:
def analyze_demographics(df, top_influencers):
    """Analyze demographic patterns among top influencers"""

    print("\n=== DEMOGRAPHIC ANALYSIS OF TOP INFLUENCERS ===")

    top_users = [user for user, _ in top_influencers]
    influencer_data = df[df['author'].isin(top_users)]

    if len(influencer_data) > 0:
        print(f"Analyzing {len(influencer_data)} top influencers...")

        # Account type distribution
        if 'account_type' in influencer_data:
            account_dist = influencer_data['account_type'].value_counts()
            print("\nAccount Types among Top Influencers:")
            for account_type, count in account_dist.items():
                print(f"  {account_type}: {count}")

        # Gender distribution
        if 'gender' in influencer_data:
            gender_dist = influencer_data['gender'].value_counts()
            print("\nGender Distribution among Top Influencers:")
            for gender, count in gender_dist.items():
                print(f"  {gender}: {count}")

        # Sentiment analysis
        if 'sentiment' in influencer_data:
            avg_sentiment = influencer_data['sentiment'].mean()
            print(f"\nAverage Sentiment among Top Influencers: {avg_sentiment:.3f}")

# Execute demographic analysis
print("Analyzing demographics...")
analyze_demographics(df, top_mention_in_degree)

#### **Notes**:

- Each node is one of the top 10 most-mentioned accounts.
- Colour distinguishes unique users.
- Node size = how many times they were mentioned.
- Edges = mutual mentions among the top 10 (visibility interactions).
- Numeric labels = exact mention counts (shows relative dominance).

Mentions were chosen because, they represent active engagement, recognition by peers, amplification potential, and measurable authority — all of which are central to what it means to be influential in a social network.



# **Question Two**